# 🔀 Notebook 03 — Hybrider, kontekst og fairness

**ALS gir Lea bedre anbefalinger. Men fungerer det for *alle*?**

Nå går vi fra modell til produkt. Vi kombinerer langtidsprofil med korttidskontekst
og sjekker om systemet faktisk er godt nok for flere enn mainstream-brukerne.

## Hvorfor étt verktøy aldri er nok

Du bruker ikke bare en hammer til å bygge et hus. Samme med anbefalingssystemer —
ingen enkeltmodell løser hele produktproblemet:

- **Popularitet** er robust, men upersonlig
- **Innholdsbasert** hjelper ved nye items og tynne signaler
- **ALS** gir sterk personalisering når det finnes data
- **Kontekst og reranking** hjelper når brukerens situasjon eller produktkrav endrer seg

Hybrider lar oss kombinere disse styrkene.

## Oppsett

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import lil_matrix, diags
from src.recommenders.als import ALSRecommender
from src.recommenders.popularity import PopularityRecommender
from src.data import load_interactions, load_item_metadata, load_sessions, get_genre_matrix
from src.split import leave_one_out_split, build_sparse_matrix, session_train_test_split
from src.metrics import recall_at_k, coverage, novelty, intra_list_similarity
from src.fairness import popularity_bias_analysis, genre_calibration_score, group_recall_comparison
from src.rerank import mmr_rerank

interactions = load_interactions()
items = load_item_metadata()
sessions = load_sessions()
train_df, test_df = leave_one_out_split(interactions)
n_users = interactions.user_id.max() + 1
n_items = interactions.item_id.max() + 1
train_matrix = build_sparse_matrix(train_df, n_users, n_items)
genre_matrix = get_genre_matrix(items)
user_ids = test_df['user_id'].values
test_items = test_df['item_id'].values
K = 10
LEA_ID = 451

als = ALSRecommender(factors=64, regularization=0.01, iterations=15, random_state=42)
als.fit(train_matrix, show_progress=True)
recs_als = als.recommend(user_ids, train_matrix, K)
item_pop_counts = np.asarray(train_matrix.sum(axis=0)).flatten()
item_pop_frac = item_pop_counts / n_users

print('Oppsett ferdig')

## 🏋️ Oppgave 4 — Korttidskontekst som hybridsignal

### Hva er en sesjon?

En sesjon er en sammenhengende rekke handlinger — for eksempel filmene Lea ser
i løpet av en kveld. Rekkefølgen betyr noe: hvis hun ser en thriller og deretter
en annen thriller, sier det noe om hva hun er i humør for *akkurat nå*.

### Hvorfor trenger vi sesjonskontekst?

ALS kjenner Leas langtidsprofil — hvem hun er generelt. Men den vet ikke
at hun akkurat nå sitter i sofaen og vil ha spenning, ikke drama.
Sesjonskontekst fanger opp dette korttidshumøret.

### Hvordan fungerer det?

Vi teller: etter film A, hva ser folk typisk etterpå? Det gir oss en
**overgangsmatrise** — en slags «hva kommer neste»-tabell. Kombinert
med ALS-profilen gir den oss det beste fra to verdener.

In [ ]:
sess_sizes = sessions.groupby('session_id').size()
print(f'Antall sesjoner: {len(sess_sizes):,}')
print(f'Sesjonslengde — min: {sess_sizes.min()}, median: {sess_sizes.median():.0f}, maks: {sess_sizes.max()}')

lea_sessions = sessions[sessions.user_id == LEA_ID]
lea_session_ids = lea_sessions.session_id.unique()
print(f'Lea har {len(lea_session_ids)} sesjoner')

### Bygg overgangsmatrise fra sesjoner

Vi teller hvor ofte item A etterfølges av item B i samme sesjon,
og normaliserer til sannsynligheter.

In [ ]:
train_sessions, test_sessions = session_train_test_split(sessions, interactions)
counts = lil_matrix((n_items, n_items), dtype=np.float64)
sorted_sessions = train_sessions.sort_values(['session_id', 'position'])
prev_item, prev_session = -1, -1
for _, row in sorted_sessions.iterrows():
    session_id, item_id = row['session_id'], row['item_id']
    if session_id == prev_session and prev_item >= 0:
        counts[prev_item, item_id] += 1
    prev_item, prev_session = item_id, session_id
transition = counts.tocsr()
row_sums = np.asarray(transition.sum(axis=1)).flatten()
row_sums[row_sums == 0] = 1.0
transition = diags(1.0 / row_sums) @ transition
print(f'Overgangsmatrise: {transition.shape}, nnz={transition.nnz:,}')

### Blend ALS-profil med sesjonskontekst

Vi blander to signaler: ALS sin langtidsprofil (hvem er Lea generelt?)
med sesjonens korttidssignal (hva ser hun akkurat nå?).

`lambda_` styrer blandingen: 1.0 = bare ALS, 0.0 = bare sesjon.
Ingen av delene alene er nok — bare ALS ignorerer øyeblikket,
bare sesjon glemmer hvem brukeren er.

In [ ]:
def recommend_session_blend(als_model, transition, test_sessions,
                           train_matrix, k=10, lambda_=0.7):
    dense_transition = transition.toarray()
    recommendations, ground_truth = [], []
    for _, group in test_sessions.groupby('session_id'):
        group = group.sort_values('position')
        if len(group) < 2:
            continue
        user_id = group['user_id'].iloc[0]
        context_items = group['item_id'].values[:-1].tolist()
        target_item = group['item_id'].values[-1]

        als_scores = als_model.user_factors[user_id] @ als_model.item_factors.T
        als_scores = (als_scores - als_scores.min()) / max(als_scores.max() - als_scores.min(), 1e-10)

        session_scores = np.zeros(dense_transition.shape[0])
        weight = 1.0
        # Bare de 3 siste filmene i sesjonen: korttidshumør er ferskvare,
        # eldre kontekst er allerede fanget av ALS-langtidsprofilen.
        # weight *= 0.8 gir geometrisk forfall (1.0, 0.8, 0.64) slik at
        # det aller siste klikket teller mest. Begge er fornuftige startverdier
        # man kan tune — prøv gjerne -5 eller decay=0.9.
        for item_id in reversed(context_items[-3:]):
            if 0 <= item_id < dense_transition.shape[0]:
                session_scores += weight * dense_transition[item_id]
            weight *= 0.8
        if session_scores.max() > 0:
            session_scores = session_scores / session_scores.max()

        blended = lambda_ * als_scores + (1 - lambda_) * session_scores
        seen = set(train_matrix[user_id].indices) | set(context_items)
        for seen_item in seen:
            if 0 <= seen_item < len(blended):
                blended[seen_item] = -np.inf
        recommendations.append(np.argsort(-blended)[:k])
        ground_truth.append(target_item)
    return np.array(recommendations), np.array(ground_truth)

In [ ]:
for lambda_value in [0.0, 0.7, 1.0]:
    recs, targets = recommend_session_blend(als, transition, test_sessions,
                                           train_matrix, k=K, lambda_=lambda_value)
    print(f'lambda={lambda_value:.1f}: Recall@{K}={recall_at_k(recs, targets, K):.4f}')

> 💬 **Diskuter**
>
> 1. Hvorfor kan en kombinasjon av langtidsprofil og sesjon være bedre enn bare én av delene?
> 2. Hvem hjelper korttidskontekst mest?
> 3. Hva sier dette om hvorfor ekte systemer blir hybride?

## 🏋️ Oppgave 5 — Fairness og reranking

Til nå har vi målt Recall som et gjennomsnitt over alle brukere. Men et gjennomsnitt
kan skjule at modellen fungerer bra for mainstream-brukere og dårlig for nisjeprofiler.
Fairness handler om å måle *hvem* som tjener — og hvem som taper.

> *Amira (CTO):* «Høy gjennomsnittlig Recall er fint for en rapport.
> Men fortell meg dette: fungerer systemet like godt for brukerne
> som ser obskure dokumentarer som for dem som ser alt på forsiden?»

Se på grafen under — hvem tjener, og hvem taper?

In [ ]:
bias_df = popularity_bias_analysis(recs_als, item_pop_counts, k=K)
group_df = group_recall_comparison(recs_als, test_items, train_matrix, item_pop_counts, K)

print(bias_df)

fig, ax = plt.subplots(figsize=(10, 5))
width = 0.35
x = range(len(bias_df))
ax.bar([idx - width / 2 for idx in x], bias_df['share_catalogue'], width, label='Andel i katalog')
ax.bar([idx + width / 2 for idx in x], bias_df['share_recommended'], width, label='Andel i anbefalinger')
ax.set_xticks(list(x))
ax.set_xticklabels(bias_df['bin'], rotation=20, ha='right')
ax.set_ylabel('Andel')
ax.set_title('Popularitetsbias: katalog vs anbefalinger')
ax.legend()
plt.tight_layout()
plt.show()

print('Recall@K per brukergruppe (ALS):')
print(group_df.to_string(index=False))

### MMR — Maximal Marginal Relevance

MMR er en re-rankeringsalgoritme som balanserer **relevans** mot **mangfold**.

Tenk på det som å lage en spilleliste: du vil ikke ha 10 like sanger på rad,
selv om de alle scorer høyt. MMR velger den neste filmen som er både relevant
*og* forskjellig fra det du allerede har valgt.

**Prosessen steg for steg:**

1. Start med en tom liste og et sett kandidater (f.eks. topp 100 fra ALS)
2. For hver gjenværende kandidat, beregn: λ · relevans − (1−λ) · likhet med det du allerede valgte
3. Velg kandidaten med høyest score, legg den til listen
4. Gjenta til listen har K filmer

`lambda_` styrer tradeoffet: λ = 1.0 betyr «bare relevans» (ingen diversitet),
λ = 0.3 betyr «prioriter mangfold tungt». Vi tester flere verdier:

In [ ]:
def apply_mmr(als_model, user_ids, train_matrix, genre_matrix, k=10, lambda_=0.5, n_cand=100):
    candidate_ids, candidate_scores = als_model.model.recommend(user_ids, train_matrix[user_ids], N=n_cand, filter_already_liked_items=True)
    recommendations = np.zeros((len(user_ids), k), dtype=np.int32)
    for row_index in range(len(user_ids)):
        recommendations[row_index] = mmr_rerank(candidate_ids[row_index], candidate_scores[row_index], genre_matrix, k=k, lambda_=lambda_)
    return recommendations

# Vis tradeoff-gradienten: lambda_ styrer balansen relevans ↔ diversitet
print(f"{'Modell':<16} {'Recall@10':>10} {'ILS (↓=bedre)':>14}")
print('-' * 44)
for lam in [1.0, 0.7, 0.5, 0.3]:
    if lam == 1.0:
        recs = recs_als
        label = 'ALS (ingen MMR)'
    else:
        recs = apply_mmr(als, user_ids, train_matrix, genre_matrix, k=K, lambda_=lam)
        label = f'ALS+MMR λ={lam}'
    r = recall_at_k(recs, test_items, K)
    ils = intra_list_similarity(recs, genre_matrix, K)
    print(f'{label:<16} {r:>10.4f} {ils:>14.4f}')

# Vi tar med λ=0.5 videre som en balansert default (omtrent like deler relevans og mangfold).
# Tabellen over viser hele gradienten — 0.4–0.7 er typiske produksjonsverdier.
recs_mmr = apply_mmr(als, user_ids, train_matrix, genre_matrix, k=K, lambda_=0.5)

**ILS** (intra-list similarity) måler gjennomsnittlig genre-likhet *innenfor* en brukers topp-10.
Lavere ILS = listene inneholder mer sjangermessig variasjon.

`coverage` og `novelty` er katalog-aggregater på tvers av alle brukere — de fanger ikke
within-user diversitet. ILS er det riktige målet her, og det beveger seg tydelig med λ.

Re-ranking for mangfold er en **eksplisitt tradeoff**: litt lavere Recall mot
lister brukeren faktisk opplever som bredere. `lambda_` er parameteren du skrur på.

> 💬 **Diskuter**
>
> 1. Hvem taper på en ren ALS-modell?
> 2. Hvilken `lambda_`-verdi ville du valgt for StreamNord? Hvorfor?
> 3. Hva ville du fortalt Amira om tradeoffet?

## 🔍 Når modeller feiler

Til nå har hver modell vært strengt bedre enn den forrige.
Men det er ikke hele sannheten. La oss se nærmere.

In [ ]:
# Finnes det brukere der ALS gjør det VERRE enn popularitet?
pop_rec = PopularityRecommender().fit(train_matrix)
recs_pop = pop_rec.recommend(user_ids, train_matrix, K)

pop_hits = np.array([1 if test_items[i] in recs_pop[i] else 0 for i in range(len(user_ids))])
als_hits = np.array([1 if test_items[i] in recs_als[i] else 0 for i in range(len(user_ids))])

pop_wins = np.sum((pop_hits == 1) & (als_hits == 0))
als_wins = np.sum((als_hits == 1) & (pop_hits == 0))
print(f'Brukere der popularitet finner riktig, men ALS bommer: {pop_wins}')
print(f'Brukere der ALS finner riktig, men popularitet bommer: {als_wins}')
print(f'\n→ ALS er bedre i gjennomsnitt, men popularitet vinner for {pop_wins} brukere.')

> *Amira:* «Så ingen modell er universelt best.
> Det er derfor vi trenger hybrider — og det er derfor vi måler på grupper, ikke bare gjennomsnitt.»

### Lærdom

- En modell med høyere gjennomsnittlig Recall kan likevel **tape** for spesifikke brukersegmenter
- Popularitet vinner ofte for brukere med mainstream-smak og lite historikk
- ALS kan overfit til majoritetsmønstre og bomme på nisjeprofiler
- **Tradeoffs er uunngåelige** — spørsmålet er hvem du prioriterer

## 📊 Produksjonsfairness — Monitorering og vedlikehold

Hybrider og re-ranking hjelper offline. Men når systemet går live,
trenger Amira et dashboard som sier fra hvis noe sklir.

**Tre metrikker vi tracker ukentlig:**

**1. Coverage (dekning)**
Hvor stor andel av katalogen dukker opp i minst én anbefaling?
Lav coverage betyr at systemet «glemmer» store deler av innholdet.

**2. Eksponering (Gini-koeffisient)**
Er eksponeringen jevnt fordelt, eller dominerer noen få items?
Gini = 0 betyr perfekt jevn fordeling, Gini = 1 betyr alt konsentrert på én.

**3. Recall-gap mellom brukergrupper**
Er systemet like godt for mainstream-brukere som for nisje-brukere?
Vi segmenterer brukere etter populariteten i deres *historikk* (ikke anbefalinger).

- Hvis Recall for nisjeprofiler dropper > 5% fra forrige uke, alert!

In [ ]:
# FAIRNESS DASHBOARD — Hva Amira sjekker hver uke i produksjon

def fairness_dashboard(recommendations, test_items, train_matrix, user_ids, item_pop_frac, genre_matrix, k=10):
    """Simuler ukentlig fairness check i produksjon."""
    n_items = len(item_pop_frac)
    
    # Metrikk 1: Unik dekke (hvor mange ulike items anbefales?)
    unique_items = len(np.unique(recommendations))
    coverage_pct = (unique_items / n_items) * 100
    
    # Metrikk 2: Gini-koeffisient (0 = jevn fordeling, 1 = alt konsentrert)
    item_counts = np.bincount(recommendations.flatten(), minlength=n_items)
    item_counts = item_counts[item_counts > 0]
    sorted_counts = np.sort(item_counts)  # ascending for standard Gini
    n = len(item_counts)
    index = np.arange(1, n + 1)
    gini = (2 * np.sum(index * sorted_counts)) / (n * np.sum(sorted_counts)) - (n + 1) / n
    
    # Metrikk 3: Recall per brukergruppe (mainstream vs niche basert på historikk)
    user_bias = np.array([
        np.mean(item_pop_frac[train_matrix[uid].indices])
        if len(train_matrix[uid].indices) > 0 else 0
        for uid in user_ids
    ])
    mainstream_users = np.where(user_bias > np.median(user_bias))[0]
    niche_users = np.where(user_bias <= np.median(user_bias))[0]
    
    recall_mainstream = recall_at_k(recommendations[mainstream_users], test_items[mainstream_users], k) if len(mainstream_users) > 0 else 0
    recall_niche = recall_at_k(recommendations[niche_users], test_items[niche_users], k) if len(niche_users) > 0 else 0
    
    return {
        'coverage_pct': coverage_pct,
        'gini_coefficient': gini,
        'recall_mainstream': recall_mainstream,
        'recall_niche': recall_niche,
        'recall_gap': abs(recall_mainstream - recall_niche)
    }

# Kjør dashboard for ALS og ALS+MMR
print('FAIRNESS DASHBOARD')
print('=' * 60)
for model_name, recs in [('ALS (rent)', recs_als), ('ALS+MMR (re-ranked)', recs_mmr)]:
    metrics = fairness_dashboard(recs, test_items, train_matrix, user_ids, item_pop_frac, genre_matrix, K)
    print(f'\n{model_name}:')
    print(f'  Coverage: {metrics["coverage_pct"]:.1f}% av katalog får minst 1 anbefaling')
    print(f'  Gini: {metrics["gini_coefficient"]:.3f} (0=jevnt, 1=konsentrert)')
    print(f'  Recall (mainstream users): {metrics["recall_mainstream"]:.4f}')
    print(f'  Recall (niche users): {metrics["recall_niche"]:.4f}')
    print(f'  Recall-gap: {metrics["recall_gap"]:.4f} ← Bør være < 0.03')
    
    if metrics['recall_gap'] > 0.03:
        print('  ⚠️  ALERT: Gapet mellom brukergrupper for stort!')
    if metrics['gini_coefficient'] > 0.6:
        print('  ⚠️  ALERT: Få items dominerer — mangfold i fare!')


### Handlingsplan

**Dersom coverage dropper under 20%:** Hybrid med popularitet eller content-based

**Dersom Gini overstiger 0.6:** Øk MMR lambda eller legg til diversitet-bias

**Dersom recall-gap > 0.05:** Slice data — retrain separate modeller for nisjeprofiler

**Dersom nye items aldri anbefales:** Fallback til innholdsbasert for items < 14 dager gamle

> 💬 **Diskuter:** Hva ville du prioritert dersom kun én av disse metrikken kan forbedres?

---

> *Amira:* «Bra. Nå har dere vist at hybrider kan hjelpe, og at fairness må måles
> *og* vedlikeholdes. Jeg er mer trygg på at dette systemet kan gå i produksjon.
> Men det er én ting vi ikke har testet: hva skjer med alt dette når brukeren er helt ny?
> Og hva anbefaler dere faktisk at StreamNord shipper?»

**Neste steg** → `04_ship_decision.ipynb`

## Valgfri appendix — kalibrering

Hvis dere vil gå dypere i fairness, kan dere måle om anbefalingene matcher
brukerens faktiske sjangerprofil.

In [ ]:
cal_als, cal_mmr = [], []
for row_index, user_id in enumerate(user_ids):
    cal_als.append(genre_calibration_score(user_id, recs_als[row_index], train_matrix, genre_matrix))
    cal_mmr.append(genre_calibration_score(user_id, recs_mmr[row_index], train_matrix, genre_matrix))

print('Kalibrerings-KL (lavere = bedre):')
print(f'  ALS:     {np.mean(cal_als):.4f}')
print(f'  ALS+MMR: {np.mean(cal_mmr):.4f}')